# Lab 1: ML Metadata (MLMD) — Enhanced Walkthrough

## Tracking a Real Sklearn Pipeline with Lineage Visualization

In this lab, you will learn how to use **ML Metadata (MLMD)** to record and retrieve metadata
associated with a real machine learning workflow. Unlike a basic walkthrough that only registers
placeholder artifacts, this lab:

1. **Trains a real scikit-learn model** (Random Forest on the Iris dataset)
2. **Records every stage** — data ingestion, preprocessing, training, and evaluation — in an MLMD metadata store
3. **Tracks lineage** between datasets, transformed data, trained models, and evaluation metrics
4. **Visualizes the full lineage DAG** so you can trace any artifact back to its origins

### Prerequisites
```bash
pip install ml-metadata scikit-learn matplotlib networkx
```

### MLMD Data Model Refresher

| Concept | Description |
|---|---|
| **ArtifactType** | Schema for a category of artifacts (e.g., Dataset, Model) |
| **Artifact** | A specific instance — a file, a model, a metric result |
| **ExecutionType** | Schema for a pipeline step (e.g., Trainer, Evaluator) |
| **Execution** | A specific run of that step |
| **Event** | Links an Artifact to an Execution as INPUT or OUTPUT |
| **Context** | Groups artifacts & executions (e.g., an Experiment) |

---
## Part 1 — Setup & Imports

In [5]:
# Install dependencies (uncomment if needed)
# !pip install ml-metadata scikit-learn matplotlib networkx

In [10]:
import os
import json
import datetime
import warnings
warnings.filterwarnings('ignore')

# MLMD imports
from ml_metadata import metadata_store
from ml_metadata.proto import metadata_store_pb2

# Sklearn imports
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib

# Visualization imports
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

print('Setup complete!')

Setup complete!


In [13]:
import ml_metadata
print(ml_metadata.__version__)
print(dir(ml_metadata))

AttributeError: module 'ml_metadata' has no attribute '__version__'

---
## Part 2 — Initialize the Metadata Store

We use a **fake (in-memory) database** for fast experimentation.
In production, you would swap this for SQLite or MySQL.

In [12]:
import ml_metadata as mlmd
from ml_metadata.proto import metadata_store_pb2

connection_config = metadata_store_pb2.ConnectionConfig()
connection_config.fake_database.SetInParent()
store = mlmd.MetadataStore(connection_config)
print('Metadata store initialized (in-memory).')

AttributeError: module 'ml_metadata' has no attribute 'MetadataStore'

---
## Part 3 — Register Artifact Types

We define four artifact types for our pipeline:
- **RawDataset** — the original data before any processing
- **TransformedDataset** — data after preprocessing (scaling, splitting)
- **Model** — a trained sklearn model saved to disk
- **ModelMetrics** — evaluation metrics (accuracy, F1, etc.)

In [ ]:
# --- RawDataset ---
raw_data_type = metadata_store_pb2.ArtifactType()
raw_data_type.name = 'RawDataset'
raw_data_type.properties['name'] = metadata_store_pb2.STRING
raw_data_type.properties['num_samples'] = metadata_store_pb2.INT
raw_data_type.properties['num_features'] = metadata_store_pb2.INT
raw_data_type_id = store.put_artifact_type(raw_data_type)

# --- TransformedDataset ---
transformed_data_type = metadata_store_pb2.ArtifactType()
transformed_data_type.name = 'TransformedDataset'
transformed_data_type.properties['name'] = metadata_store_pb2.STRING
transformed_data_type.properties['split'] = metadata_store_pb2.STRING
transformed_data_type.properties['num_samples'] = metadata_store_pb2.INT
transformed_data_type_id = store.put_artifact_type(transformed_data_type)

# --- Model ---
model_type = metadata_store_pb2.ArtifactType()
model_type.name = 'Model'
model_type.properties['name'] = metadata_store_pb2.STRING
model_type.properties['framework'] = metadata_store_pb2.STRING
model_type.properties['version'] = metadata_store_pb2.INT
model_type.properties['hyperparameters'] = metadata_store_pb2.STRING
model_type_id = store.put_artifact_type(model_type)

# --- ModelMetrics ---
metrics_type = metadata_store_pb2.ArtifactType()
metrics_type.name = 'ModelMetrics'
metrics_type.properties['accuracy'] = metadata_store_pb2.DOUBLE
metrics_type.properties['f1_weighted'] = metadata_store_pb2.DOUBLE
metrics_type.properties['report'] = metadata_store_pb2.STRING
metrics_type_id = store.put_artifact_type(metrics_type)

print('Registered Artifact Types:')
for at in store.get_artifact_types():
    print(f'  - {at.name} (id={at.id})')

---
## Part 4 — Register Execution Types

We define three execution types matching our pipeline steps:
1. **DataIngestion** — loading the raw dataset
2. **Preprocessing** — splitting and scaling
3. **Training** — fitting the model and evaluating

In [ ]:
def register_execution_type(store, name):
    """Helper to register an ExecutionType with a 'state' property."""
    exec_type = metadata_store_pb2.ExecutionType()
    exec_type.name = name
    exec_type.properties['state'] = metadata_store_pb2.STRING
    type_id = store.put_execution_type(exec_type)
    return type_id

ingestion_type_id = register_execution_type(store, 'DataIngestion')
preprocessing_type_id = register_execution_type(store, 'Preprocessing')
training_type_id = register_execution_type(store, 'Training')

print('Registered Execution Types:')
for et in store.get_execution_types():
    print(f'  - {et.name} (id={et.id})')

---
## Part 5 — Helper Functions

These utility functions reduce boilerplate when creating executions and events.

In [ ]:
def create_execution(store, type_id, state='RUNNING'):
    """Create and register an execution, returning its ID."""
    execution = metadata_store_pb2.Execution()
    execution.type_id = type_id
    execution.properties['state'].string_value = state
    [execution_id] = store.put_executions([execution])
    return execution_id


def complete_execution(store, execution_id, type_id):
    """Mark an execution as COMPLETED."""
    execution = metadata_store_pb2.Execution()
    execution.id = execution_id
    execution.type_id = type_id
    execution.properties['state'].string_value = 'COMPLETED'
    store.put_executions([execution])


def register_event(store, artifact_id, execution_id, event_type):
    """Record an event (INPUT or OUTPUT) linking an artifact to an execution."""
    event = metadata_store_pb2.Event()
    event.artifact_id = artifact_id
    event.execution_id = execution_id
    event.type = event_type
    store.put_events([event])


print('Helper functions defined.')

---
## Part 6 — Pipeline Step 1: Data Ingestion

We load the Iris dataset and register it as a **RawDataset** artifact.
An **DataIngestion** execution is created and linked via events.

In [ ]:
# ---- Actual ML work ----
iris = load_iris()
X, y = iris.data, iris.target

print(f'Loaded Iris dataset: {X.shape[0]} samples, {X.shape[1]} features')

# ---- MLMD: Create execution ----
ingestion_exec_id = create_execution(store, ingestion_type_id)

# ---- MLMD: Register the raw dataset artifact ----
raw_data_artifact = metadata_store_pb2.Artifact()
raw_data_artifact.uri = 'in-memory://sklearn.datasets.load_iris'
raw_data_artifact.type_id = raw_data_type_id
raw_data_artifact.properties['name'].string_value = 'Iris Dataset'
raw_data_artifact.properties['num_samples'].int_value = X.shape[0]
raw_data_artifact.properties['num_features'].int_value = X.shape[1]
[raw_data_id] = store.put_artifacts([raw_data_artifact])

# ---- MLMD: Register output event ----
register_event(store, raw_data_id, ingestion_exec_id,
               metadata_store_pb2.Event.DECLARED_OUTPUT)

# ---- MLMD: Complete execution ----
complete_execution(store, ingestion_exec_id, ingestion_type_id)

print(f'Data Ingestion complete. Raw dataset artifact ID: {raw_data_id}')

---
## Part 7 — Pipeline Step 2: Preprocessing

We split the data into train/test sets and apply `StandardScaler`.
This step takes the **RawDataset** as input and produces two
**TransformedDataset** artifacts (train split + test split).

In [ ]:
# ---- Actual ML work ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train set: {X_train_scaled.shape[0]} samples')
print(f'Test set:  {X_test_scaled.shape[0]} samples')

# ---- MLMD: Create execution ----
preprocess_exec_id = create_execution(store, preprocessing_type_id)

# ---- MLMD: Input event (raw data -> preprocessing) ----
register_event(store, raw_data_id, preprocess_exec_id,
               metadata_store_pb2.Event.DECLARED_INPUT)

# ---- MLMD: Register transformed train artifact ----
train_artifact = metadata_store_pb2.Artifact()
train_artifact.uri = 'in-memory://iris_train_scaled'
train_artifact.type_id = transformed_data_type_id
train_artifact.properties['name'].string_value = 'Iris Train (scaled)'
train_artifact.properties['split'].string_value = 'train'
train_artifact.properties['num_samples'].int_value = X_train_scaled.shape[0]
[train_artifact_id] = store.put_artifacts([train_artifact])

# ---- MLMD: Register transformed test artifact ----
test_artifact = metadata_store_pb2.Artifact()
test_artifact.uri = 'in-memory://iris_test_scaled'
test_artifact.type_id = transformed_data_type_id
test_artifact.properties['name'].string_value = 'Iris Test (scaled)'
test_artifact.properties['split'].string_value = 'test'
test_artifact.properties['num_samples'].int_value = X_test_scaled.shape[0]
[test_artifact_id] = store.put_artifacts([test_artifact])

# ---- MLMD: Output events ----
register_event(store, train_artifact_id, preprocess_exec_id,
               metadata_store_pb2.Event.DECLARED_OUTPUT)
register_event(store, test_artifact_id, preprocess_exec_id,
               metadata_store_pb2.Event.DECLARED_OUTPUT)

# ---- MLMD: Complete execution ----
complete_execution(store, preprocess_exec_id, preprocessing_type_id)

print(f'Preprocessing complete. Train artifact ID: {train_artifact_id}, Test artifact ID: {test_artifact_id}')

---
## Part 8 — Pipeline Step 3: Training & Evaluation

We train a **RandomForestClassifier**, evaluate it on the test set,
and register both the **Model** and **ModelMetrics** artifacts.

Notice how hyperparameters are stored as a JSON string property on the Model artifact.

In [ ]:
# ---- Actual ML work ----
hyperparams = {'n_estimators': 100, 'max_depth': 5, 'random_state': 42}
clf = RandomForestClassifier(**hyperparams)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred, target_names=iris.target_names)

print(f'Accuracy: {acc:.4f}')
print(f'F1 (weighted): {f1:.4f}')
print(f'\nClassification Report:\n{report}')

# Save model to disk
os.makedirs('artifacts', exist_ok=True)
model_path = 'artifacts/iris_rf_model.joblib'
joblib.dump(clf, model_path)
print(f'Model saved to {model_path}')

In [ ]:
# ---- MLMD: Create training execution ----
training_exec_id = create_execution(store, training_type_id)

# ---- MLMD: Input events (train + test data -> training) ----
register_event(store, train_artifact_id, training_exec_id,
               metadata_store_pb2.Event.DECLARED_INPUT)
register_event(store, test_artifact_id, training_exec_id,
               metadata_store_pb2.Event.DECLARED_INPUT)

# ---- MLMD: Register model artifact ----
model_artifact = metadata_store_pb2.Artifact()
model_artifact.uri = model_path
model_artifact.type_id = model_type_id
model_artifact.properties['name'].string_value = 'Iris RandomForest v1'
model_artifact.properties['framework'].string_value = 'scikit-learn'
model_artifact.properties['version'].int_value = 1
model_artifact.properties['hyperparameters'].string_value = json.dumps(hyperparams)
[model_artifact_id] = store.put_artifacts([model_artifact])

# ---- MLMD: Register metrics artifact ----
metrics_artifact = metadata_store_pb2.Artifact()
metrics_artifact.uri = 'in-memory://eval_metrics_v1'
metrics_artifact.type_id = metrics_type_id
metrics_artifact.properties['accuracy'].double_value = acc
metrics_artifact.properties['f1_weighted'].double_value = f1
metrics_artifact.properties['report'].string_value = report
[metrics_artifact_id] = store.put_artifacts([metrics_artifact])

# ---- MLMD: Output events ----
register_event(store, model_artifact_id, training_exec_id,
               metadata_store_pb2.Event.DECLARED_OUTPUT)
register_event(store, metrics_artifact_id, training_exec_id,
               metadata_store_pb2.Event.DECLARED_OUTPUT)

# ---- MLMD: Complete execution ----
complete_execution(store, training_exec_id, training_type_id)

print(f'Training complete.')
print(f'  Model artifact ID:   {model_artifact_id}')
print(f'  Metrics artifact ID: {metrics_artifact_id}')

---
## Part 9 — Group Everything Under an Experiment Context

We create an **Experiment** context and associate all artifacts and executions with it.
This lets us query "everything that happened in this experiment" later.

In [ ]:
# ---- Register ContextType ----
experiment_ctx_type = metadata_store_pb2.ContextType()
experiment_ctx_type.name = 'Experiment'
experiment_ctx_type.properties['note'] = metadata_store_pb2.STRING
experiment_ctx_type.properties['timestamp'] = metadata_store_pb2.STRING
experiment_ctx_type_id = store.put_context_type(experiment_ctx_type)

# ---- Create Context instance ----
experiment_ctx = metadata_store_pb2.Context()
experiment_ctx.type_id = experiment_ctx_type_id
experiment_ctx.name = 'iris-classification-exp-1'
experiment_ctx.properties['note'].string_value = (
    'Iris classification with RandomForest, StandardScaler preprocessing'
)
experiment_ctx.properties['timestamp'].string_value = (
    datetime.datetime.now().isoformat()
)
[experiment_ctx_id] = store.put_contexts([experiment_ctx])

# ---- Attributions (artifacts -> context) ----
attributions = []
for aid in [raw_data_id, train_artifact_id, test_artifact_id,
            model_artifact_id, metrics_artifact_id]:
    attr = metadata_store_pb2.Attribution()
    attr.artifact_id = aid
    attr.context_id = experiment_ctx_id
    attributions.append(attr)

# ---- Associations (executions -> context) ----
associations = []
for eid in [ingestion_exec_id, preprocess_exec_id, training_exec_id]:
    assoc = metadata_store_pb2.Association()
    assoc.execution_id = eid
    assoc.context_id = experiment_ctx_id
    associations.append(assoc)

store.put_attributions_and_associations(attributions, associations)

print(f'Experiment context created: "{experiment_ctx.name}" (id={experiment_ctx_id})')
print(f'  Linked {len(attributions)} artifacts and {len(associations)} executions.')

---
## Part 10 — Querying the Metadata Store

Now that we have recorded everything, let's query the store to answer
real-world questions you might ask in production.

In [ ]:
# Question 1: What artifacts exist in this experiment?
print('='*60)
print('Artifacts in experiment:')
print('='*60)
for a in store.get_artifacts_by_context(experiment_ctx_id):
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f'  [{a_type.name}] id={a.id}  uri={a.uri}')

In [ ]:
# Question 2: Which dataset was used to train the model?
# Strategy: Start from the Model artifact, trace back through events.
print('='*60)
print('Tracing lineage: Model -> Training inputs -> Raw data')
print('='*60)

# Find the model
model = store.get_artifacts_by_type('Model')[0]
print(f'\n1. Found model: "{model.properties["name"].string_value}" (id={model.id})')

# Find the execution that produced this model
model_events = store.get_events_by_artifact_ids([model.id])
output_event = [e for e in model_events if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT][0]
training_exec = store.get_executions_by_id([output_event.execution_id])[0]
exec_type_name = store.get_execution_types_by_id([training_exec.type_id])[0].name
print(f'2. Produced by execution: "{exec_type_name}" (id={training_exec.id})')

# Find all inputs to that execution
exec_events = store.get_events_by_execution_ids([training_exec.id])
input_ids = [e.artifact_id for e in exec_events if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
input_artifacts = store.get_artifacts_by_id(input_ids)
print(f'3. Training inputs:')
for a in input_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f'   - [{a_type.name}] "{a.properties["name"].string_value}" (id={a.id})')

# Trace further back: where did the train data come from?
train_data_events = store.get_events_by_artifact_ids([input_artifacts[0].id])
output_ev = [e for e in train_data_events if e.type == metadata_store_pb2.Event.DECLARED_OUTPUT][0]
preproc_exec = store.get_executions_by_id([output_ev.execution_id])[0]
preproc_events = store.get_events_by_execution_ids([preproc_exec.id])
raw_ids = [e.artifact_id for e in preproc_events if e.type == metadata_store_pb2.Event.DECLARED_INPUT]
raw_artifacts = store.get_artifacts_by_id(raw_ids)
print(f'4. Original raw data:')
for a in raw_artifacts:
    a_type = store.get_artifact_types_by_id([a.type_id])[0]
    print(f'   - [{a_type.name}] "{a.properties["name"].string_value}" '
          f'({a.properties["num_samples"].int_value} samples)')

In [ ]:
# Question 3: What were the hyperparameters and metrics for the model?
print('='*60)
print('Model details')
print('='*60)
model = store.get_artifacts_by_type('Model')[0]
print(f'Name:            {model.properties["name"].string_value}')
print(f'Framework:       {model.properties["framework"].string_value}')
print(f'Hyperparameters: {model.properties["hyperparameters"].string_value}')

metrics = store.get_artifacts_by_type('ModelMetrics')[0]
print(f'\nAccuracy:        {metrics.properties["accuracy"].double_value:.4f}')
print(f'F1 (weighted):   {metrics.properties["f1_weighted"].double_value:.4f}')

---
## Part 11 — Lineage Visualization

This is the key addition over the base lab. We build a **directed acyclic graph (DAG)**
from all events in the metadata store and render it with `networkx` + `matplotlib`.

The graph shows:
- **Blue rectangles** = Artifacts
- **Orange rounded boxes** = Executions
- **Arrows** = data flow (INPUT edges go into executions, OUTPUT edges come out)

In [ ]:
def build_lineage_graph(store):
    """
    Build a NetworkX DiGraph from all artifacts, executions, and events
    in the metadata store.
    """
    G = nx.DiGraph()
    
    # Add artifact nodes
    for a in store.get_artifacts():
        a_type = store.get_artifact_types_by_id([a.type_id])[0]
        name = a.properties.get('name')
        label = name.string_value if name and name.string_value else a.uri
        split = a.properties.get('split')
        if split and split.string_value:
            label += f'\n({split.string_value})'
        node_id = f'a_{a.id}'
        G.add_node(node_id, label=f'{a_type.name}\n{label}',
                   node_type='artifact', artifact_type=a_type.name)
    
    # Add execution nodes
    for e in store.get_executions():
        e_type = store.get_execution_types_by_id([e.type_id])[0]
        node_id = f'e_{e.id}'
        state = e.properties['state'].string_value if 'state' in e.properties else ''
        G.add_node(node_id, label=f'{e_type.name}\n[{state}]',
                   node_type='execution')
    
    # Add edges from events
    all_artifacts = store.get_artifacts()
    all_artifact_ids = [a.id for a in all_artifacts]
    all_events = store.get_events_by_artifact_ids(all_artifact_ids)
    
    for ev in all_events:
        a_node = f'a_{ev.artifact_id}'
        e_node = f'e_{ev.execution_id}'
        if ev.type in (metadata_store_pb2.Event.INPUT,
                       metadata_store_pb2.Event.DECLARED_INPUT):
            G.add_edge(a_node, e_node)
        elif ev.type in (metadata_store_pb2.Event.OUTPUT,
                         metadata_store_pb2.Event.DECLARED_OUTPUT):
            G.add_edge(e_node, a_node)
    
    return G


def visualize_lineage(G, figsize=(16, 10)):
    """
    Render the lineage DAG with distinct styles for artifacts and executions.
    """
    # Use a layered layout for DAG
    for layer, nodes in enumerate(nx.topological_generations(G)):
        for node in nodes:
            G.nodes[node]['subset'] = layer
    pos = nx.multipartite_layout(G, subset_key='subset', align='horizontal')
    
    # Separate node types
    artifact_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'artifact']
    exec_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'execution']
    
    labels = nx.get_node_attributes(G, 'label')
    
    fig, ax = plt.subplots(1, 1, figsize=figsize)
    
    # Draw artifact nodes (blue squares)
    nx.draw_networkx_nodes(G, pos, nodelist=artifact_nodes,
                           node_color='#4A90D9', node_shape='s',
                           node_size=3500, alpha=0.9, ax=ax)
    
    # Draw execution nodes (orange circles)
    nx.draw_networkx_nodes(G, pos, nodelist=exec_nodes,
                           node_color='#F5A623', node_shape='o',
                           node_size=3500, alpha=0.9, ax=ax)
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, edge_color='#555555',
                           arrows=True, arrowsize=20,
                           connectionstyle='arc3,rad=0.1', ax=ax)
    
    # Draw labels
    nx.draw_networkx_labels(G, pos, labels, font_size=7,
                            font_color='white', font_weight='bold', ax=ax)
    
    # Legend
    legend_elements = [
        mpatches.Patch(facecolor='#4A90D9', label='Artifact'),
        mpatches.Patch(facecolor='#F5A623', label='Execution'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=11)
    ax.set_title('ML Pipeline Lineage DAG (from MLMD)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('artifacts/lineage_dag.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Lineage DAG saved to artifacts/lineage_dag.png')


print('Visualization functions defined.')

In [ ]:
# Build and render the lineage graph
G = build_lineage_graph(store)
visualize_lineage(G)

print(f'\nGraph summary: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

---
## Part 12 — Lineage Trace: From Model Back to Raw Data

This function recursively walks **backward** through events to find
every upstream artifact that contributed to a given artifact.
This is the core capability that makes MLMD valuable in production:
if a model goes wrong, you can trace exactly which data and steps produced it.

In [ ]:
def trace_lineage_back(store, artifact_id, depth=0, visited=None):
    """
    Recursively trace an artifact's lineage back to its origins.
    Returns a list of (depth, artifact_type_name, artifact_name, artifact_id) tuples.
    """
    if visited is None:
        visited = set()
    if artifact_id in visited:
        return []
    visited.add(artifact_id)
    
    results = []
    artifact = store.get_artifacts_by_id([artifact_id])[0]
    a_type = store.get_artifact_types_by_id([artifact.type_id])[0]
    name = artifact.properties.get('name')
    a_name = name.string_value if name and name.string_value else artifact.uri
    results.append((depth, a_type.name, a_name, artifact.id))
    
    # Find executions that produced this artifact
    events = store.get_events_by_artifact_ids([artifact_id])
    for ev in events:
        if ev.type in (metadata_store_pb2.Event.OUTPUT,
                       metadata_store_pb2.Event.DECLARED_OUTPUT):
            # Find inputs to that execution
            exec_events = store.get_events_by_execution_ids([ev.execution_id])
            for exec_ev in exec_events:
                if exec_ev.type in (metadata_store_pb2.Event.INPUT,
                                    metadata_store_pb2.Event.DECLARED_INPUT):
                    results.extend(
                        trace_lineage_back(store, exec_ev.artifact_id, depth+1, visited)
                    )
    return results


# Trace the model's full lineage
print('Full lineage of the trained model:')
print('='*60)
model = store.get_artifacts_by_type('Model')[0]
lineage = trace_lineage_back(store, model.id)

for depth, type_name, name, aid in lineage:
    indent = '  ' * depth
    connector = 'ROOT' if depth == 0 else '<--'
    print(f'{indent}{connector} [{type_name}] "{name}" (id={aid})')

---
## Part 13 — Summary

In this enhanced lab, you:

1. **Initialized** an MLMD metadata store
2. **Registered** artifact types (RawDataset, TransformedDataset, Model, ModelMetrics) and execution types (DataIngestion, Preprocessing, Training)
3. **Ran a real sklearn pipeline** — loaded Iris data, scaled it, trained a RandomForest, and evaluated it
4. **Recorded every step** as MLMD executions with proper input/output events
5. **Grouped everything** under an Experiment context
6. **Queried the store** to answer questions like "which data trained this model?"
7. **Visualized the full lineage DAG** using networkx + matplotlib
8. **Traced lineage recursively** from the model back to the raw dataset

### Key Takeaways

- MLMD acts as a **structured log** for your ML pipeline — every artifact and step is tracked
- **Events** are the backbone of lineage — they connect artifacts to the executions that consumed or produced them
- **Contexts** let you group related work (e.g., one experiment) for easy querying
- In production, you would use **SQLite or MySQL** instead of the fake in-memory database
- The lineage DAG gives you a visual map to **debug issues** and **audit model provenance**

